# Assignment 4

Download an email attachment that matches given criteria from your own Gmail account.

1. About app passwords:

Create app password for your own Gmail account.

- https://support.google.com/mail/answer/185833?hl=en-GB

- https://support.google.com/accounts/answer/185833?hl=en


2. Create a label in your Gmail account. Name of the label should be “Reports”.

 Labels are used to organize your mails. Labels are also known as folders.

https://support.google.com/mail/answer/118708?hl=en&co=GENIE.Platform%3DDesktop


Create a rule filter so that all emails sent by you should go to “Reports” folder. Ie. Apply the label Reports to such emails.

https://support.google.com/mail/answer/6579?hl=en
 

3. Send 3 separate emails to yourself for each attached csv file. The subject of email Should be “Report for 2020-01-01”, “Report for 2020-02-01” and “Report for 2020-03-01” respectively.

Report for 2020-01-01
Report for 2020-02-01
Report for 2020-03-01

4. Write a Python code to connect to your Gmail account using your email id and app password. The code should receive 4 inputs.

- Label.
- Sender.
- Subject filter regex.
- File name regex.

The task is to programmatically download the email attachment that matches above 4 search criterions.

Eg input: 
Reprot your-email@gmail.com "Report for 2020.*" Report.*
This should download all 3 attachments from 3 emails sent by you and with the label “Reports”.

Once the attachments are successfully downloaded, make sure that all emails are marked as read. This also needs to be done programmatically.

After a successful download, print following details for each file: File name, File size, File extension, Number of rows in a file.

If no email matches with given criteria, then print the message: "No email matched with given criteria."

Submit a Jupyter notebook(.ipynb) file as a part of submission. 
 

Note 1: While submitting the code, please remove your email and app passwords from the submission files.

Note 2: While writing the code, make sure it's designed with all the best practices, like try-catch blocks, closing the connection etc. You'll need to create a class and different methods for different functionalities like connection creation, connection validation, actual file download, closing connection, printing the output and wherever else required.

Note 3: Both Subject filter and File names should be regular expressions.

In [10]:
import imaplib
import email
import os
import re
import pandas as pd

In [ ]:
EMAIL = "-@gmail.com"
APP_PASSWORD = "myyt mhpj vyop qpsu"

In [12]:
try:
    mail = imaplib.IMAP4_SSL("imap.gmail.com")
    mail.login(EMAIL, APP_PASSWORD)
    print("Connected Successfully")

except Exception as e:
    print("Connection Error:", e)

Connected Successfully


In [ ]:
label = "Reports"
sender = "-@gmail.com"
subject_regex = r"Report for 2020.*"
filename_regex = r"Report.*"

In [14]:
try:
    mail.select(label)
    print("Opened Label:", label)

except Exception as e:
    print("Label Error:", e)

Opened Label: Reports


In [15]:
try:
    status, messages = mail.search(
        None,
        f'(FROM "{sender}")'
    )
    email_ids = messages[0].split()
    print("Emails Found:", len(email_ids))

except Exception as e:
    print("Search Error:", e)

Emails Found: 3


In [19]:
matched = False
try:
    for email_id in email_ids:
        status, msg_data = mail.fetch(email_id, "(RFC822)")
        raw_email = msg_data[0][1]
        msg = email.message_from_bytes(raw_email)
        subject = msg["Subject"]

        if re.search(subject_regex, subject):
            for part in msg.walk():
                if part.get_content_disposition() == "attachment":
                    filename = part.get_filename()

                    if re.search(filename_regex, filename):
                        matched = True
                        filepath = os.path.join(os.getcwd(), filename)
                        with open(filepath, "wb") as f:
                            f.write(part.get_payload(decode=True))

                        mail.store(email_id, '+FLAGS', '\\Seen')

                        file_size = os.path.getsize(filepath)

                        extension = os.path.splitext(filename)[1]

                        extension = os.path.splitext(filename)[1].lower()

                        df = pd.read_excel(filepath)

                        rows = len(df)

                        print("\nDownloaded Successfully")
                        print("File Name:", filename)
                        print("File Size:", file_size, "bytes")
                        print("Extension:", extension)
                        print("Rows:", rows)

    if not matched:
        print("No email matched with given criteria.")

except Exception as e:
    print("Download Error:", e)


Downloaded Successfully
File Name: Report for 2020-01-01.xlsx
File Size: 9907 bytes
Extension: .xlsx
Rows: 31

Downloaded Successfully
File Name: Report for 2020-02-01.xlsx
File Size: 9831 bytes
Extension: .xlsx
Rows: 29

Downloaded Successfully
File Name: Report for 2020-03-01.xlsx
File Size: 9921 bytes
Extension: .xlsx
Rows: 31


In [20]:
try:
    mail.logout()
    print("Connection Closed")

except Exception as e:
    print("Logout Error:", e)

Connection Closed
